# OpenFold2/AlphaFold2 Ray Pipeline Example

This notebook demonstrates multi-stages ray pipeline for folding process.

## Imports

### Step 1: Setup Environment

In [1]:
# Set these environment variables to match your setup before running the notebook.
# Uncomment and modify if needed:
# %env ALPHAFOLD2_1_CKPT=/path/to/alphafold2_1.pt
# %env ENGINE_OUTPUT_DIR=/path/to/engine/output

In [2]:
import os
import time
import tempfile
from pathlib import Path
import torch
import ray

from tensorrt_bionemo.pipeline.processor.engine_proc import (
    EngineProcessorConfig, build_processor)
from tensorrt_bionemo.pipeline.stages.configs import (
    EngineStageConfig, FeatureGeneratorStageConfig, TokenizerStageConfig, ParserStageConfig, WriterStageConfig)
from tensorrt_bionemo.data.schemas import InputRequest, Polymer, MSARecord

# Set environment variables (uncomment and modify paths as needed)

# PyTorch backend only
ALPHAFOLD2_1_CKPT = os.environ.get("ALPHAFOLD2_1_CKPT", "alphafold2_1.pt")
# With TRT acceleration
ENGINE_OUTPUT_DIR = os.environ.get("ENGINE_OUTPUT_DIR", "alphafold2_1_engines")

MODEL_NAME = "alphafold2_1"
output_dir = "output"

# ── Inline sample data (T1047s1: FlgH-FlgI subunit 1, 232 residues) ──
SAMPLE_SEQUENCE = (
    "MQKNAAHTYAISSLLVLSLTGCAWIPSTPLVQGATSAQPVPGPTPVANGSIFQSAQPINYGYQPLFED"
    "RRPRNIGDTLTIVLQENVSASKSSSANASRDGKTNFGFDTVPRYLQGLFGNARADVEASGGNTFNGKGG"
    "ANASNTFSGTLTVTVDQVLVNGNLHVVGEKQIAINQGTEFIRFSGVVNPRTISGSNTVPSTQVADARI"
    "EYVGNGYINEAQNMGWLQRFFLNLSPM"
)

# Trimmed MSA: query + 5 homologs (enough for pipeline demonstration)
SAMPLE_MSA_A3M = """\
>101
MQKNAAHTYAISSLLVLSLTGCAWIPSTPLVQGATSAQPVPGPTPVANGSIFQSAQPINYGYQPLFEDRRPRNIGDTLTIVLQENVSASKSSSANASRDGKTNFGFDTVPRYLQGLFGNARADVEASGGNTFNGKGGANASNTFSGTLTVTVDQVLVNGNLHVVGEKQIAINQGTEFIRFSGVVNPRTISGSNTVPSTQVADARIEYVGNGYINEAQNMGWLQRFFLNLSPM
>A0A1B7IUJ2
----------IVMCLVLATTGCALIPTKPLVEGATTAQPVPGPAPVVNGSIFQTAQPVNYGYQPLFEDRRPRNVGDTLTIVLQENVSASKSSSANASRDGKTNFGFDVTPRYLEGLFGNNRADVDASGGNSFNGKGGANASNTFSGTLTVTVDQVLANGNLHVVGEKQIAINQGTEFIRFSGVVNPRTISGSNTVPSTQVADARIEYVGNGYINEAQNMGWLQRFFLNLSPM
>A0A2P8VQF1
MQKTGAQFNPLVMTLALALTGCAWVPSTPLVQGATTAQPVPAPAPVVNGSIFQSVQPINYGYQPLFEDRRPRNVGDTLTIQLQENVSASKSSSANASRSSSSKFGFDAVPRYLEGLFGNARADMSASGDNGFNGKGGANANNTFSGTLTVTVDQVLANGNLHVVGEKQIAINQGTEFIRFSGVVNPRTISGTNTVPSTQVADARIEYVGNGYINEAQNMGWLQRFFLNLSPM
>A0A6M8UIV5
--QNQPRRLFV-AGLLLTLNGCALIPHTPLVQGPTTAQPLPASPPVVNGSIFQGVMPMNYGYQPLFEDRRPRNIGDTLTIVLQENVSASKSSSANASRDGSSSFGLTTVPNALEGLLGGNKTALDGAGKNDFAGKGGASANNTFTGTITVTVNQVLPNGNLHVVGEKQIEINQGTEFIRFSGVVNPRTISGSNTVVSTQVADARIEYVGNGYINEAQSMGWLQRFFLNISPM
>UPI000C195E16
---------GLVAALLLTLNGCALVPRTPLVKGPTTAQPVPAQPPVTNGSIFQGVMPMNYGYQPLFEDRRPRNVGDTLTIVLQENVSASKNSSANATRNGSTSLGMSVVPRYLSGPLGNNRADIAGEGKNDFAGKGGATANNTFTGTITVTVNQVLPNGNLNVVGEKQIEINQGTEFIRFSGVVNPRTISGSNTVISTQVADARIEYVGNGYINEAQTMGWLQRFFLNLSPM
>A0A381H561
MQKYAAHHYPIMALLVVSLTGCAWIPSTPLVQGATTAQPIPGPTPVANGSIFQSAQPINYGYQPLFEDRRPRNIGDTLTIVLQENVSASKSSSANASRDGKTNFGFDTVPRYLQGLFGNARADMEASGGNSFNGKGGANASNTFSGTLTVTVDQVLANGNLHVVGEKQIAINQGTEFIRFSGVVNPRTISGSNSVPSTQVADARIEYVGNGYINEGAKYGLAATFL------
"""

# Write MSA to temp file (needed by MSARecord parser)
_msa_tmpfile = tempfile.NamedTemporaryFile(mode="w", suffix=".a3m", delete=False)
_msa_tmpfile.write(SAMPLE_MSA_A3M)
_msa_tmpfile.close()
SAMPLE_MSA_PATH = _msa_tmpfile.name
print(f"MSA written to {SAMPLE_MSA_PATH}")

MSA written to /tmp/tmpj1im8_qh.a3m


### Step 2: Build TensorRT Engines (if you want to try with TensorRT engine)

Build optimized TensorRT engines for the Evoformer module. This may take 10-30 minutes depending on your GPU.

In [3]:
# Build TensorRT engines (skipped if already built)
import subprocess
import shutil
from tensorrt_bionemo import EXAMPLES_DIR

subprocess.run('nvidia-smi', check=True)

# Conversion script from the installed wheel
CONVERT_SCRIPT = str(EXAMPLES_DIR / "openfold2" / "convert_evoformer_checkpoint.py")

# ============ TRT Build Configuration (single source of truth) ============
BUILD_CONFIG = {
    "model_name": MODEL_NAME,
    "module": "evoformer",
    "safetensor_dir": "evoformer_safetensors",
    "output_dir": ENGINE_OUTPUT_DIR,
    "dtype": "bfloat16",
    "max_seqlen": 1536,
    "min_seqlen": 16,
    "triangle_attn_backend": "CUEQUIV",
}

# Build commands derived from config
CONVERT_CMD = [
    "python", CONVERT_SCRIPT,
    "--model_name", BUILD_CONFIG["model_name"],
    "--output_dir", BUILD_CONFIG["safetensor_dir"],
    "--triangle_attn_backend", BUILD_CONFIG["triangle_attn_backend"],
    "--local_checkpoint", ALPHAFOLD2_1_CKPT,
]

BUILD_CMD = [
    "trtbnm-build",
    "--model", BUILD_CONFIG["model_name"],
    "--module", BUILD_CONFIG["module"],
    "--checkpoint_dir", BUILD_CONFIG["safetensor_dir"],
    "--max_seqlen", str(BUILD_CONFIG["max_seqlen"]),
    "--min_seqlen", str(BUILD_CONFIG["min_seqlen"]),
    "--output_dir", BUILD_CONFIG["output_dir"],
    "--weakly_dtype", BUILD_CONFIG["dtype"],
]

engine_file = os.path.join(BUILD_CONFIG["output_dir"], "trt", "rank0.engine")
if os.path.exists(engine_file):
    print(f"TRT engines already exist at: {BUILD_CONFIG['output_dir']}")
    print("   Delete the directory and re-run this cell to rebuild.")
else:
    print(f"Building TRT engines (this takes 10-30 minutes)...")
    print(f"\nConfig: {BUILD_CONFIG}\n")
    
    # Clean up previous safetensor directory if exists
    if os.path.exists(BUILD_CONFIG["safetensor_dir"]):
        shutil.rmtree(BUILD_CONFIG["safetensor_dir"])
    
    print(f"Step 1: Converting checkpoint...")
    print(f"  Command: {' '.join(CONVERT_CMD)}\n")
    subprocess.run(CONVERT_CMD, check=True)

    print(f"\nStep 2: Building TRT engines...")
    print(f"  Command: {' '.join(BUILD_CMD)}\n")
    subprocess.run(BUILD_CMD, check=True)

    print(f"\nTRT engines built successfully at: {BUILD_CONFIG['output_dir']}")

Fri Apr 10 09:00:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:01:00.0 Off |                    0 |
| N/A   33C    P0            128W /  700W |    6827MiB /  81559MiB |     28%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

2026-04-10 09:00:57,994 - tensorrt_bionemo.hubs.local - INFO - Loading alphafold2_1 from local filesystem /workspace/cache/models/alphafold2_1.pt


Total time of converting checkpoints: 00:00:01



Step 2: Building TRT engines...
  Command: trtbnm-build --model alphafold2_1 --module evoformer --checkpoint_dir evoformer_safetensors --max_seqlen 1536 --min_seqlen 16 --output_dir /workspace/cache/alphafold2_1_engines --weakly_dtype bfloat16



[04/10/2026-09:01:01] [TRT-LLM] [I] Building module evoformer for backend trt
[04/10/2026-09:01:01] [TRT-LLM] [W] Overriding # of builder profiles <= 2.
[04/10/2026-09:01:01] [TRT-LLM] [I] Module config dtype: float32, weakly_dtype: bfloat16
[04/10/2026-09:01:01] [TRT-LLM] [I] Building weakly-typed engine with dtype bfloat16.
[04/10/2026-09:01:01] [TRT-LLM] [I] Set dtype to float32.


...

[04/10/2026-09:26:09] [TRT] [I] [MemUsageStats] Peak memory usage of TRT CPU/GPU memory allocators: CPU 113 MiB, GPU 22184 MiB


[04/10/2026-09:26:10] [TRT-LLM] [I] Total time of building evoformer: 00:25:02


[04/10/2026-09:26:10] [TRT] [I] Serialized 9608 bytes of code generator cache.
[04/10/2026-09:26:11] [TRT] [I] Serialized 3524527 bytes of compilation cache.
[04/10/2026-09:26:11] [TRT] [I] Serialized 3807 timing cache entries
[04/10/2026-09:26:11] [TRT-LLM] [I] Timing cache serialized to model.cache
[04/10/2026-09:26:11] [TRT-LLM] [I] Build phase peak memory: 13790.21 MB, children: 0.00 MB
[04/10/2026-09:26:11] [TRT-LLM] [I] Serializing engine to /workspace/cache/alphafold2_1_engines/trt/rank0.engine...


[04/10/2026-09:26:12] [TRT-LLM] [I] Engine serialized. Total time: 00:00:01
[04/10/2026-09:26:12] [TRT-LLM] [I] Total time of building all engines: 00:25:11



TRT engines built successfully at: /workspace/cache/alphafold2_1_engines


---

## Helper functions

Once you have your model checkpoint and optionally built TRT engines, you can run the inference pipeline below.

In [4]:
def get_trt_accelerated_configs() -> dict:
    """Check if TRT engines are available from environment variables."""
    if os.path.exists(ENGINE_OUTPUT_DIR):
        return {"evoformer": {"checkpoint": ENGINE_OUTPUT_DIR, "backend": "trt"}}
    return None

def get_num_gpus() -> int:
    from tensorrt_bionemo.pipeline.processor.utils import get_available_gpu_count   
    return get_available_gpu_count()
    
def create_sample_requests(repeat: int = 50):
    """Create sample protein folding requests using inline data."""
    requests = []
    for i in range(repeat):
        requests.append(
            InputRequest(
                input_id=f"T1047s1_{i}",
                polymers=[Polymer(
                    chain_id="A",
                    sequence=SAMPLE_SEQUENCE,
                    msas=[MSARecord(path=SAMPLE_MSA_PATH)]
                )]
            )
        )
    return requests

def run_pipeline(
    config: EngineProcessorConfig,
    trt_acc: dict = None, 
    repeat: int = 50
):
    """
    Run the protein structure prediction pipeline.
    
    Args:
        model_name: Model to use (e.g., "alphafold2_1")
        trt_acc: TensorRT acceleration config
        num_gpu_replicas: Number of GPU replicas for the folding engine stage
    """
    
    os.environ["RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION"] = "0.5"
    
    print(f"\n{'='*80}")
    print(f"OpenFold2/AlphaFold2 Structure Prediction Pipeline")
    print(f"{'='*80}")
    print(f"Model: {config.model_source}")
    
    print(f"Config: {config}")
    
    engine_kwargs = {}
    if trt_acc:
        print(f"Backend: PyTorch + Evoformer TensorRT")
        print(f"Accelerated modules:")
        engine_kwargs["accelerated_configs"] = trt_acc
        for module, path in trt_acc.items():
            print(f"  - {module}: {path}")
    else:
        print(f"Backend: PyTorch (Standard)")
    print(f"{'='*80}\n")
    
    processor = build_processor(config)
    
    requests = create_sample_requests(repeat)
    records = [{"record": req, "__record_id": req["input_id"]} for req in requests]
    
    print(f"Processing {len(records)} protein sequences...")
    
    ds = ray.data.from_items(records)
    
    ctx = ray.data.DataContext.get_current()
    ctx.max_errored_blocks = 100
    
    ds = processor(ds)
    
    success_count = 0
    error_count = 0
    results = []
    
    # materialize the dataset (batching, for the streaming mode don't push the line)
    start_time = time.time()
    ds = ds.materialize()
    elapsed_time = time.time() - start_time
    for row in ds.iter_rows():
        err = row.get("__inference_error__", {})
        if err.get("error_msg"):
            error_count += 1
            print(f"\n  Error in {row.get('__record_id', 'unknown')}:")
            print(f"   {err.get('error_msg')}")
        else:
            success_count += 1
            output_path = row.get("output_path")
            if output_path:
                results.append({
                    "id": row.get("__record_id"),
                    "path": output_path
                })
    
    
    print(f"\n{'='*80}")
    print(f"Pipeline Results")
    print(f"{'='*80}")
    print(f"Total time: {elapsed_time:.2f} seconds")
    print(f"Time per sequence: {elapsed_time/len(records):.2f} seconds")
    print(f"Successful: {success_count}/{len(records)}")
    print(f"Errors: {error_count}/{len(records)}")
    
    if results:
        print(f"\nOutput structures:")
        for result in results[:10]:
            print(f"  {result['id']}: {result['path']}")
        print(f"...")
    
    print(f"{'='*80}\n")
    
    return success_count, error_count, elapsed_time

## Configuration and Setup

Set up the model name and output directory. Make sure the required environment variables are set before running the pipeline.

## Run the Pipeline

Execute the pipeline with the configured settings.

In [5]:
import logging
import warnings
# First, get the handle for the logger you want to modify
ray_data_logger = logging.getLogger("ray.data")
ray_serve_logger = logging.getLogger("ray.serve")
ray_data_logger.setLevel(logging.ERROR)
ray_serve_logger.setLevel(logging.ERROR)
os.environ["TLLM_LOG_LEVEL"] = "ERROR"

warnings.filterwarnings("ignore", category=DeprecationWarning)

### 1. With single GPU

In [6]:
from ray.data import DataContext
# Disable all progress bars
ray.shutdown()
DataContext.get_current().enable_progress_bars = False
print(f"Run on single GPU - {torch.cuda.get_device_name(0)}")
num_gpus = get_num_gpus()
print(f"Detected {num_gpus} GPU(s)")
trt_acc = get_trt_accelerated_configs() # Set to None to run only torch backend
try:
    config = EngineProcessorConfig(
        model_source=MODEL_NAME,
        executor_backend="ray",
        parser_stage=ParserStageConfig(compute=2),
        tokenizer_stage=TokenizerStageConfig(compute=2, num_cpus=2),
        feature_generator_stage=FeatureGeneratorStageConfig(compute=4, num_cpus=4),
        writer_stage=WriterStageConfig(
            compute=2,
            output_path=output_dir,
            format="pdb"
        ),
        engine_stage=EngineStageConfig(
            compute=1,
            num_cpus=4
        )
    )
    success, errors, elapsed = run_pipeline(config, trt_acc, repeat=5)
    
    if errors == 0:
        print(f"Pipeline completed successfully!")
    else:
        print(f"Pipeline completed with {errors} errors")
        
except Exception as e:
    print(f"\nPipeline failed with error:")
    print(f"   {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

Run on single GPU - NVIDIA H100 80GB HBM3
Detected 1 GPU(s)

OpenFold2/AlphaFold2 Structure Prediction Pipeline
Model: alphafold2_1
Config: batch_size=1 accelerator_type=None concurrency=1 model_source='alphafold2_1' runtime_env=None max_pending_requests=None max_concurrent_batches=8 should_continue_on_error=False executor_backend='ray' engine_kwargs={} parser_stage=ParserStageConfig(enabled=True, batch_size=None, compute=2, runtime_env=None, num_cpus=None, memory=None, compute_by_rows=True, drop_keys=None) tokenizer_stage=TokenizerStageConfig(enabled=True, batch_size=None, compute=2, runtime_env=None, num_cpus=2.0, memory=None, compute_by_rows=True, drop_keys=None) feature_generator_stage=FeatureGeneratorStageConfig(enabled=True, batch_size=None, compute=4, runtime_env=None, num_cpus=4.0, memory=None, compute_by_rows=True, drop_keys=None, init_context=None) writer_stage=WriterStageConfig(enabled=True, batch_size=None, compute=2, runtime_env=None, num_cpus=None, memory=None, compute_by

2026-04-10 09:26:24,060	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Processing 5 protein sequences...


(MapWorker(MapBatches(TokenizerUDF)) pid=672843) /usr/local/lib/python3.12/dist-packages/ray/_common/utils.py:95: DeprecationWarning: There is no current event loop
(MapWorker(MapBatches(TokenizerUDF)) pid=672843)   return asyncio.get_event_loop_policy().get_event_loop()


...

(MapWorker(MapBatches(FoldingEngineUDF)) pid=672852) /usr/local/lib/python3.12/dist-packages/cuequivariance_ops_torch/triangle_attention.py:165: UserWarning: Non-SM100f kernel expects bias to be float32 so it's going to be cast to torch.float32. Check if you can change your code for maximum performance.
(MapWorker(MapBatches(FoldingEngineUDF)) pid=672852)   warnings.warn(
(pid=gcs_server) [2026-04-10 09:26:53,131 E 671952 671952] (gcs_server) gcs_server.cc:961: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


(raylet) [2026-04-10 09:26:53,210 E 672620 672620] (raylet) main.cc:1047: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


(MapWorker(MapBatches(FoldingEngineUDF)) pid=672852) /usr/local/lib/python3.12/dist-packages/cuequivariance_ops_torch/triangle_attention.py:165: UserWarning: Non-SM100f kernel expects bias to be float32 so it's going to be cast to torch.float32. Check if you can change your code for maximum performance.
(MapWorker(MapBatches(FoldingEngineUDF)) pid=672852)   warnings.warn(


(_StatsActor pid=672841) [2026-04-10 09:26:56,785 E 672841 672965] core_worker_process.cc:863: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=672846) /usr/local/lib/python3.12/dist-packages/ray/_common/utils.py:95: DeprecationWarning: There is no current event loop
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=672846)   return asyncio.get_event_loop_policy().get_event_loop()
[2026-04-10 09:26:57,007 E 209 672837] core_worker_process.cc:863: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14



Pipeline Results
Total time: 34.96 seconds
Time per sequence: 6.99 seconds
Successful: 5/5
Errors: 0/5

Output structures:
  T1047s1_0: output/T1047s1_0.pdb
  T1047s1_1: output/T1047s1_1.pdb
  T1047s1_2: output/T1047s1_2.pdb
  T1047s1_3: output/T1047s1_3.pdb
  T1047s1_4: output/T1047s1_4.pdb
...

Pipeline completed successfully!


### 2. With multiple GPUs (Replica mode)

In [7]:
from ray.data import DataContext
# Disable all progress bars
ray.shutdown()
DataContext.get_current().enable_progress_bars = False
print("Run with replica mode using all GPUs")
trt_acc = get_trt_accelerated_configs() # Set to None to run only torch backend
try:
    config = EngineProcessorConfig.create_default_replica_mode_config(MODEL_NAME, output_dir)
    success, errors, elapsed = run_pipeline(config, trt_acc, repeat=5)
    
    if errors == 0:
        print(f"Pipeline completed successfully!")
    else:
        print(f"Pipeline completed with {errors} errors")
        
except Exception as e:
    print(f"\nPipeline failed with error:")
    print(f"   {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

/usr/lib/python3.12/subprocess.py:1127: ResourceWarning: subprocess 672064 is still running
  _warn("subprocess %s is still running" % self.pid,
/usr/local/lib/python3.12/dist-packages/ray/_private/node.py:1075: ResourceWarning: unclosed file <_io.TextIOWrapper name='/dev/null' mode='w' encoding='UTF-8'>
  process_info = ray._private.services.start_gcs_server(


Run with replica mode using all GPUs

OpenFold2/AlphaFold2 Structure Prediction Pipeline
Model: alphafold2_1
Config: batch_size=1 accelerator_type=None concurrency=1 model_source='alphafold2_1' runtime_env=None max_pending_requests=None max_concurrent_batches=8 should_continue_on_error=True executor_backend='ray' engine_kwargs={} parser_stage=ParserStageConfig(enabled=True, batch_size=None, compute=1, runtime_env=None, num_cpus=None, memory=None, compute_by_rows=True, drop_keys=None) tokenizer_stage=TokenizerStageConfig(enabled=True, batch_size=None, compute=1, runtime_env=None, num_cpus=4.0, memory=None, compute_by_rows=True, drop_keys=None) feature_generator_stage=FeatureGeneratorStageConfig(enabled=True, batch_size=None, compute=1, runtime_env=None, num_cpus=8.0, memory=None, compute_by_rows=True, drop_keys=None, init_context=None) writer_stage=WriterStageConfig(enabled=True, batch_size=None, compute=1, runtime_env=None, num_cpus=1.0, memory=None, compute_by_rows=True, drop_keys=Non

/usr/local/lib/python3.12/dist-packages/ray/_private/utils.py:381: ResourceWarning: unclosed file <_io.TextIOWrapper name='/sys/fs/cgroup/cpu.max' mode='r' encoding='UTF-8'>
  max_file = open(cpu_max_file_name).read()


2026-04-10 09:27:14,932	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Processing 5 protein sequences...


...

(MapWorker(MapBatches(FoldingEngineUDF)) pid=676775) /usr/local/lib/python3.12/dist-packages/cuequivariance_ops_torch/triangle_attention.py:165: UserWarning: Non-SM100f kernel expects bias to be float32 so it's going to be cast to torch.float32. Check if you can change your code for maximum performance.
(MapWorker(MapBatches(FoldingEngineUDF)) pid=676775)   warnings.warn(


(MapWorker(MapBatches(FoldingEngineUDF)) pid=676775) /usr/local/lib/python3.12/dist-packages/cuequivariance_ops_torch/triangle_attention.py:165: UserWarning: Non-SM100f kernel expects bias to be float32 so it's going to be cast to torch.float32. Check if you can change your code for maximum performance.
(MapWorker(MapBatches(FoldingEngineUDF)) pid=676775)   warnings.warn(


(pid=gcs_server) [2026-04-10 09:27:43,953 E 675901 675901] (gcs_server) gcs_server.cc:961: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2026-04-10 09:27:44,064 E 676548 676548] (raylet) main.cc:1047: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


(_StatsActor pid=676771) [2026-04-10 09:27:47,633 E 676771 676865] core_worker_process.cc:863: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
[2026-04-10 09:27:47,827 E 209 676764] core_worker_process.cc:863: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14



Pipeline Results
Total time: 33.71 seconds
Time per sequence: 6.74 seconds
Successful: 5/5
Errors: 0/5

Output structures:
  T1047s1_0: output/T1047s1_0.pdb
  T1047s1_1: output/T1047s1_1.pdb
  T1047s1_2: output/T1047s1_2.pdb
  T1047s1_3: output/T1047s1_3.pdb
  T1047s1_4: output/T1047s1_4.pdb
...

Pipeline completed successfully!
